# Roof Edge Polyline Tracing with Gradient Validation & Snapping

[Open in Google Colab][colab] · [Open in Colab Enterprise][colab-ent] · [View on GitHub][github]

Solar panel layout design, storm-damage property underwriting, and 3D urban reconstruction require
structural roof geometry (`RIDGE`, `EAVE`, `HIP`, `VALLEY`, `RAKE` polylines). From street level,
trees frequently occlude roofs, and unconstrained vision-language model polyline predictions can
drift off physical edges or hallucinate lines along horizontal wall siding.

This tutorial ranks upward-pitched panoramic views from `capture_id` rosettes, screens out
foliage-occluded views before calling Gemini (`views.occlusion_screen`), traces candidate roof
polylines at `HIGH` media resolution (`schemas.RoofEdges`), runs a deterministic OpenCV gradient
pipeline to **validate**, **snap**, and **reject** proposed edges (`roof.validate_roof_edges`), and
cross-checks dominant eave angles in Gemini's **agentic** code-execution sandbox.

| Stage | Tool | Responsibility |
|---|---|---|
| **1. Discovery** | BigQuery (`rosette_sql`) | Find nearby rosettes (`<= 60 m`) & target bearings |
| **2. Screen** | `svi_geo.views` | Rank upward roof views (`< 1%` black) and screen trees |
| **3. Trace** | Gemini (`RoofEdges`) | Propose typed roof polylines at `HIGH` media resolution |
| **4. Snap & LSD** | `roof` + Code Exec | Reject decoys, snap to Sobel/LSD, verify eave angle |

> **Runtime & Cost Banner:** Expected runtime **<= 5 min** on Colab CPU · Gemini ceiling
> **<= $0.10** (`<= 4` plain calls + `1` code-exec call) · Cold BigQuery scan **1.83 GB** at
> **$6.25/TB** (~$0.011 first run, **0 B** when cached for up to 35 days).

[colab]: https://colab.research.google.com/github/googlemaps-samples/insights-samples
[colab-ent]: https://console.cloud.google.com/vertex-ai/colab/import
[github]: https://github.com/googlemaps-samples/insights-samples/tree/main/street_view_insights

## 1. Architecture, Data Model & Division of Labour

**Problem.** Select clear street-level views of a target building's roof, trace visible roof edges
(`RIDGE`, `EAVE`, `HIP`, `VALLEY`, `RAKE`), and filter out hallucinated or occluded segments using
deterministic gradient support and sky-contact gates (the **valid**, **snap**, and **reject**
stages).

**Data model & BigQuery cost.** Rosettes are keyed by `capture_id` (`pano_id` is nullable
attribution metadata). A cold scan of `rosette_sql` is **1.83 GB** at **$6.25/TB** on-demand
(cached locally up to 35 days so warm re-runs bill **0 B**).

**Division of labour.** BigQuery finds nearby rosettes and target bearings; Python ranks
upward-pitched roof views, screens foliage occlusion, and validates, snaps, and rejects proposed
polylines against Sobel and LSD image gradients; Vertex AI Gemini proposes `RoofEdges`; one
**agentic** vision cell runs OpenCV LSD in the sandbox to measure the dominant eave angle.

## 2. Terms of Service, Attribution & Authentication

**Terms of use & attribution.** Imagery comes from your Street View Insights dataset and is
governed by the terms of your Google Cloud agreement: do not redistribute frames or rendered crops
outside your organisation, keep the `"Imagery © Google"` attribution on every figure and map, and
do not persist raw imagery or `gcs_uri` values beyond short-lived working caches (`<= 24 h` for
frames, `<= 35 days` for snapshot-keyed metadata).

- **Colab:** calls `google.colab.auth.authenticate_user()`.
- **Local Jupyter:** run `gcloud auth application-default login` once.
- **Workstation helpers:** set `SVI_USE_GCLOUD_TOKEN=1` and `SVI_ECP_PROXY_URL` when required.
- **IAM roles needed:** `roles/bigquery.jobUser`, `roles/bigquery.dataViewer`,
  `roles/storage.objectViewer`, and `roles/aiplatform.user`.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# @title Step 1: Install the svi_geo Helper Package
# Install the shared `svi_geo` helper package (geometry, sequencing, dedup, Gemini client).
# Dependency ranges in pyproject.toml are compatible with Colab preinstalled versions.
# In a clone of the repo the local copy is used: $SVI_GEO_LOCAL_PATH, or the first parent
# directory that contains street_view_insights/panoramic/svi_geo. Otherwise pip installs it
# from GitHub at SVI_GEO_REF.
import importlib.util
import os
import pathlib
import subprocess
import sys

# TODO(after-merge): set to "main" once svi_geo is merged. Override with $SVI_GEO_REF.
SVI_GEO_REF = os.environ.get("SVI_GEO_REF", "f4087e9")


def find_local_svi_geo():
    if os.environ.get("SVI_GEO_LOCAL_PATH"):
        return os.environ["SVI_GEO_LOCAL_PATH"]
    here = pathlib.Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "street_view_insights/panoramic/svi_geo" / "pyproject.toml").is_file():
            return str(d / "street_view_insights/panoramic/svi_geo")
    return None


LOCAL_SVI_GEO = find_local_svi_geo()
if importlib.util.find_spec("svi_geo") is None:
    if LOCAL_SVI_GEO:
        src = f"{LOCAL_SVI_GEO}[notebooks]"
    else:
        src = (
            "svi-geo[notebooks] @ git+https://github.com/googlemaps-samples/insights-samples"
            f"@{SVI_GEO_REF}#subdirectory=street_view_insights/panoramic/svi_geo"
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", src])

## 3. Environment Configuration & High-Resolution Roof Tracing Settings

Fine polyline vertex localization benefits from higher visual token density, so we configure
`VIEW_W, VIEW_H = 1200, 900` and `MEDIA_RESOLUTION = "HIGH"` while keeping `N_VIEWS = 2` and
`MAX_USD = 0.10`. We also define `EDGE_COLORS` to style `RIDGE`, `EAVE`, `HIP`, `VALLEY`, and
`RAKE` polylines consistently.

In [ ]:
# @title Step 2: Configure Parameters & Initialize Cloud Clients
import json
import os

import matplotlib.pyplot as plt
from google.cloud import storage
from IPython.display import Image, display

from svi_geo import (
    attribution,
    auth,
    config,
    data,
    images,
    maps,
    rosette,
    schemas,
    usecases,
    views,
)
from svi_geo import gemini_client as gc

try:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
except ImportError:
    pass

PROJECT_ID = os.environ.get("PROJECT_ID", "YOUR_PROJECT_ID")
DATASET_ID = "imagery_insights___us"
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")
SNAPSHOT_ID = None
INCLUDE_UNPUBLISHED_PANOS = True
MODEL = gc.DEFAULT_MODEL
LOCATION = "global"
THINKING_LEVEL = "LOW"
MEDIA_RESOLUTION = "HIGH"
ENABLE_CODE_EXECUTION = True
MAX_GEMINI_CALLS = 15
MAX_USD = 0.10
CONCURRENCY = 8
SEED = 0
LAT, LNG = 28.05047, -81.96015
RADIUS_M = 60.0
N_VIEWS = 2
VIEW_W, VIEW_H = 1200, 900
EDGE_COLORS = {
    "RIDGE": (220, 50, 47),
    "EAVE": (38, 139, 210),
    "HIP": (42, 161, 152),
    "VALLEY": (108, 113, 196),
    "RAKE": (181, 137, 0),
}
OUT_DIR = pathlib.Path(os.environ.get("OUT_DIR", "~/tmp/svi_qa/uc4_roof")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)
QA_DIR = (
    pathlib.Path(LOCAL_SVI_GEO or ".") / "data" / "qa" / "2026-10-01" if LOCAL_SVI_GEO else OUT_DIR
)

settings = config.resolve_settings(PROJECT_ID, GCS_BUCKET, env=dict(os.environ), dataset=DATASET_ID)
PROJECT_ID, BUCKET = settings.project, settings.bucket
creds = auth.get_credentials()
TABLE = data.pano_table(PROJECT_ID, DATASET_ID)
SNAP_TABLE = f"{PROJECT_ID}.{DATASET_ID}.snapshots"
bq = data.QueryRunner(
    data.make_bigquery_client(PROJECT_ID, creds),
    allowed_tables=data.pano_tables(PROJECT_ID, DATASET_ID),
    cache_dir=data.DEFAULT_QUERY_CACHE,
)
fetcher = images.GcsImageFetcher(storage.Client(project=PROJECT_ID, credentials=creds))
runner = gc.GeminiRunner(
    gc.VertexGeminiBackend(
        gc.make_vertex_client(PROJECT_ID, LOCATION, creds),
        model=MODEL,
        thinking_level=THINKING_LEVEL,
        media_resolution=MEDIA_RESOLUTION,
    ),
    max_calls=MAX_GEMINI_CALLS,
    max_usd=MAX_USD,
    concurrency=CONCURRENCY,
)
intr = rosette.load_intrinsics()

## 4. Discover Nearby Panoramic Rosettes in BigQuery

We run `snapshot_catalog_sql` (`2 KB`) and `rosette_sql` (`1.83 GB` cold scan, `0 B` when cached)
within `RADIUS_M = 60.0 m` of `(LAT, LNG)` and attach camera-to-building distances and bearings via
`data.attach_target_framing`.

In [ ]:
# @title Step 3: Query Nearby Rosettes & Attach Target Bearings
snap_df = bq.run(data.snapshot_catalog_sql(SNAP_TABLE), [])
ROSETTE_SQL = data.rosette_sql(TABLE)
print(ROSETTE_SQL[:340] + "\n...")
params = data.rosette_params(
    lat=LAT,
    lng=LNG,
    radius_m=RADIUS_M,
    snapshot_id=SNAPSHOT_ID,
    include_unpublished=INCLUDE_UNPUBLISHED_PANOS,
)
rosettes = data.attach_target_framing(
    bq.run(ROSETTE_SQL, params, template_name="rosette_sql"), tlat=LAT, tlng=LNG
)
snap_info = data.summarize_snapshots(rosettes, snap_df)
print(snap_info["summary_line"])
frames = data.frames_from_rosettes(rosettes, bucket=BUCKET)
cost_bq = bq.last_cost()
snap_ids = ",".join(s[:8] for s in snap_info["snapshot_ids"]) or "none"
null_cnt = int(rosettes["pano_id"].isna().sum())
print(
    f"snapshots={snap_ids} rosettes={len(rosettes)} null_pano_id={null_cnt} "
    f"sequences={rosettes['seq_id'].nunique()} bq_bytes={cost_bq['bytes']} "
    f"bq_cached={int(cost_bq['cached'])}"
)

## 5. Rank Upward-Pitched Roof Views & Screen Tree Canopy Occlusion

Rather than using a fixed pitch angle, `views.rank_roof_views` computes an adaptive upward camera
pitch from the camera-to-target distance so the roof sits cleanly in the upper-middle band of the
perspective crop (`< 1%` analytic black border). `views.occlusion_screen` then checks sky-contact
and Excess Green foliage ratios on the rendered crop so Gemini is only called on unoccluded roof
views.

In [ ]:
# @title Step 4: Rank Upward-Pitched Roof Views & Screen Foliage Occlusion
VARIANT = usecases.DEFAULT_VARIANT
print(VARIANT.describe())
cand_df = views.rank_roof_views(
    frames, LAT, LNG, intr, n=2 * N_VIEWS, aspect=VIEW_W / VIEW_H, max_dist_m=RADIUS_M
)
roof_views, chosen, screen_records = usecases.uc4_select_views(
    frames,
    LAT,
    LNG,
    fetcher.fetch,
    intr,
    n_views=N_VIEWS,
    width=VIEW_W,
    height=VIEW_H,
    max_dist_m=RADIUS_M,
    variant=VARIANT,
)
if not chosen:
    raise RuntimeError("No unoccluded roof views passed views.occlusion_screen.")
scr0 = views.occlusion_screen(chosen[0]["image"], views.roof_box(chosen[0], chosen[0]["view"]))
print(f"worked_roof_view candidates={len(cand_df)} chosen={len(chosen)} screen={scr0}")

## 6. Structured Roof Polyline Tracing with Gemini

We inspect `usecases.UC4_PROMPT_V0` and the `schemas.RoofEdges` Pydantic schema (each `RoofEdge`
has `edge_type`, normalized `points` polyline coordinates in `0..1000`, and `confidence`), run a
single-view demonstration on `chosen[0]["image"]`, and then execute `usecases.uc4_run` to trace and
validate roof edges across the selected views.

In [ ]:
# @title Step 5: Trace Candidate Roof Polylines with Gemini
print("Prompt:", usecases.UC4_PROMPT_V0)
print("Schema keys:", list(schemas.RoofEdges.model_json_schema()["properties"]))
demo_rf = await runner.ask(
    [usecases.UC4_PROMPT_V0, chosen[0]["image"]], schemas.RoofEdges, seed=SEED
)
print(f"single_view_demo_edges={len(demo_rf.edges)}")
out = await usecases.uc4_run(chosen, runner, width=VIEW_W, height=VIEW_H, variant=VARIANT)
runner.check(context="UC4 roof edge tracing")
results = out["results"]
assert out["accepted_edges"] >= 1

## 7. Agentic Vision Verification: Measuring Dominant Eave Angles with LSD

In this **agentic** vision step (`usecases.uc4_measure_roof_angles` with `code_execution=True`),
Gemini runs OpenCV's Line Segment Detector (`LSD`) inside its Python sandbox on the roof crop to
measure the dominant eave orientation angle (`eave_angle_deg`) and cross-checks it against local
`cv2.fitLine` geometry on the validated roof edges (`code_exec_agreement=agree(...)`).

In [ ]:
# @title Step 6: Measure Eave Orientation in the Gemini Code Execution Sandbox
rbox0 = views.roof_box(chosen[0], chosen[0]["view"])
ce = await usecases.uc4_measure_roof_angles(
    chosen[0]["image"],
    runner,
    roof_box=rbox0,
    seed=SEED,
    thinking_level=THINKING_LEVEL,
    code_execution=True,
    validator=lambda r: None,
    expect_stdout=usecases.MEASURE_STDOUT_PATTERN,
)
gc.check_code_exec_trace(ce["trace"], expect_stdout=usecases.MEASURE_STDOUT_PATTERN)
assert ce["agreement_line"].startswith("code_exec_agreement=")
print(ce["agreement_line"])
ce_path = OUT_DIR / "agentic_roof_angles.png"
fig_ce, ax_ce = plt.subplots(figsize=(5, 4))
ax_ce.imshow(ce["overlay"][:, :, ::-1])
ax_ce.set_title(ce["agreement_line"])
ax_ce.axis("off")
attribution.save_figure(fig_ce, ce_path)
plt.close(fig_ce)
display(Image(filename=str(ce_path)))

## 8. Deterministic Gradient Validation, Edge Snapping & Decoy Rejection

`roof.validate_roof_edges` post-processes every Gemini-proposed polyline in three deterministic
stages:

1. **Reject (`rejected_edges`)**: Filters out segments that fall in the sky, cross foliage, or lie
   on lower-wall horizontal siding decoys.
2. **Snap**: Searches perpendicular to each surviving segment to snap vertices onto the strongest
   local Sobel/LSD image gradient ridge.
3. **Validate (`valid_edges`)**: Reports `mean_gradient_support` along accepted polylines and
   compares against a Monte Carlo random-segment baseline (`random_acceptance`).

In [ ]:
# @title Step 7: Inspect Validated vs Rejected Roof Edges & Gradient Support
r0 = results[0]
print(
    f"black_fraction_max={out['black_fraction_max']:.4f} "
    f"dark_pixel_max={out['dark_pixel_max']:.4f} "
    f"redaction_overlap_max={out['redaction_overlap_max']:.4f}"
)
rej_n = sum(len(r.rejected_edges) for r in results)
print(f"accepted_edges={out['accepted_edges']} rejected_edges={rej_n}")
print(
    f"mean_gradient_support={r0.mean_gradient_support:.2f} "
    f"random_acceptance={r0.random_acceptance:.2f}"
)

## 9. Interactive Roof Sighting Map & Accepted/Rejected Edge Overlay Grid

We render `maps.uc4_roof_map` (`map.html`) showing the selected roof camera viewpoints and sighting
rays, and `maps.render_overlay_grid` (`overlay_grid.png`) drawing accepted, gradient-snapped roof
polylines in green and rejected decoy/sky segments in dashed red with their rejection reason.

In [ ]:
# @title Step 8: Render Roof Sighting Map & QA Edge Overlay Grid
m = maps.uc4_roof_map(
    chosen, target_lat=LAT, target_lng=LNG, results=results, out_path=OUT_DIR / "map.html"
)
panels = []
for c, res_i in zip(chosen, results, strict=True):
    _ = [EDGE_COLORS[e.edge_type or "EAVE"] for e in res_i.valid_edges]
    panels.append(
        {
            "title": f"{c['capture_id'][:8]}_cam{c['cam_k']}",
            "image": c["image"],
            "accepted_edges": [e.points for e in res_i.valid_edges],
            "rejected_edges": [
                (e.points, getattr(e, "reason", None) or "low_grad_or_sky")
                for e in res_i.rejected_edges
            ],
        }
    )
grid_path = OUT_DIR / "overlay_grid.png"
maps.render_overlay_grid(
    panels,
    grid_path,
    notebook_name="roof_edge_tracing",
    qa_dir=QA_DIR,
)
display(Image(filename=str(grid_path)))

## 10. Token Scaling Preview & Cost Audit

We preview multimodal token counts across `LOW`, `MEDIUM`, and `HIGH` media resolutions via
`gc.preview_tokens`, verify total Gemini spend against `MAX_USD = $0.10`, and export
`metrics.json`.

In [ ]:
# @title Step 9: Preview Token Scaling & Export Cost Telemetry
prev_by_res = {
    lvl: gc.preview_tokens(
        runner.backend, [usecases.UC4_PROMPT_V0, chosen[0]["image"]], media_resolution=lvl
    )
    for lvl in ("LOW", "MEDIUM", "HIGH")
}
est = gc.estimate_cost(len(chosen), preview=prev_by_res[MEDIA_RESOLUTION])
print(f"preview_tokens={prev_by_res} pre_run_est_usd=${est:.4f} ceiling=${MAX_USD:.2f}")
print(runner.cost.summary(ceiling=MAX_USD))
(OUT_DIR / "metrics.json").write_text(
    json.dumps(
        {
            "notebook": "roof_edge_tracing",
            "rosettes": int(len(rosettes)),
            "null_pano_id": int(rosettes["pano_id"].isna().sum()),
            "accepted_edges": int(out["accepted_edges"]),
            "est_cost_usd": float(runner.cost.usd),
        },
        indent=2,
    ),
    encoding="utf-8",
)
artefacts = len(list(OUT_DIR.glob("*")))
print(f"map_rendered=1 artefacts={artefacts} peak_rss_mb={images.peak_rss_mb():.0f}")

## 11. Summary, Generated Artifacts & Next Steps

### Summary & Generated Artifacts

- **Pipeline Recap**: You ranked upward-pitched roof views from nearby `capture_id` rosettes,
  screened out foliage-occluded angles (`views.occlusion_screen`), traced candidate roof polylines
  at `HIGH` media resolution, ran deterministic gradient validation, snapping, and decoy rejection
  (`roof.validate_roof_edges`), and verified eave orientation in Gemini's **agentic**
  code-execution sandbox.
- **Saved Artifacts in `OUT_DIR`**: `map.html` (roof sighting map), `overlay_grid.png` (accepted
  green vs rejected red roof edges), `agentic_roof_angles.png` (sandbox LSD eave-angle check), and
  `metrics.json`.
- **Production Design Note**: Heavy tree canopy or shallow street setbacks can hide upper roof
  ridges from street level; `occlusion_screen` and `validate_roof_edges` reject unsupported
  segments rather than guessing.

### Next Steps

- Review [`svi_geo/EVALUATION.md`](../svi_geo/EVALUATION.md) for multi-AOI wall-decoy rejection and
  edge-retention benchmarks across `lakeland_fl`, `salt_lake_ut`, and `osaka_jp`.
- Revisit **[Multi-View Building Discovery](house_image_discovery_with_cost.ipynb)** and
  **[Sequential Street View Asset Detection](analyze_sequential_images.ipynb)** to combine building
  facade attributes, roof edges, and right-of-way infrastructure in a unified GIS workflow.